# Welcome to Foundations of TEM Data Management and Analysis 2026: A Hands-on CITEAM Workshop One-day program 

The 2026 CITEAM workshop is geared toward cyberinfrastructure (CI) training for the community of scientists and researchers using the new generation of Transmission Electron Microscope (TEM) instruments at UMD. The training materials presented during the workshop and the associated training activities will help reduce barriers to the use of the instruments, data, software tools, and computing to help users in key steps critical to their scientific inquiry - from data collection, data analysis, to storing and publishing data - encompassing end-to-end TEM Data Management.

This edition of the workshop includes a keynote talk from Professor Ichiro Takeuchi, and cover foundational materials relevant to the instrument users.

The day-long workshop will be structured around a mix of lectures introducing the materials customized for the instrument users, hands-on sessions where participants will have the opportunity to go through end-to-end examples with the workshop instructors, and a site visit to the actual instrument. The instructors will be available this afternoon at "office hours" to address specific questions and concerns.

-------------------------

## Section 0.1.1 - First Steps

In this introductory section, we will open a single TIFF micrograph and a stack of TIFF frames, inspect basic image properties, and generate a quick JPEG thumbnail for browsing and sharing. This will be the first step in virtually all of your microscopy workflows.

We'll take each part of the process step by step so we can see exactly how things work.

The first step is to tell the Python interpreter that we're going to use several "packages" from both the standard Python library and from the "PyPy" repository of over four thousand Open Source packages from a multitude of authors.


In [ ]:
# First, import the modules we'll be using later. All of this Python code was written
# by a multitude of other people and will do an enormous amount of work "behind the scenes".

# These are some of the standard packages used by most Python programs
import os, re, json, hashlib
from datetime import datetime
from pathlib import Path

# The next packages are specialized ones for math, drawing, and handling image files.
import numpy as np
import matplotlib.pyplot as plt
import tifffile as tiff
#import h5py   # Used for HDF5, if we decide to actually use that.
from scipy import ndimage
from scipy.signal import fftconvolve

#Finally, Hyperspy is a package that provides algorithms specifically for microscopy
import hyperspy.api as hs

#for interactive use inside Jupyter output cells:
from ipywidgets import interact, widgets
%matplotlib widget

print("done.")

### Let's set some variables to the filenames where our images are stored.

We're going to be referring to some of our input files quite a few times. We could type out those filenames in our code every time, but that is both annoying and error-prone. Instead, we'll create some variables and set them equal to the filenames. Then any time we want, we can just use the variable (with its much shorter name). If the files are moved to another location for some reason, we only have to change the names here.

You may have to scroll side-to-side to see the whole filename, by the way.

***This is a good place to mention the Data Management topic later in the day - coding metadata into monster filenames.***

***This is also a good place to talk about named parameters in the mkdir function.***

In [ ]:
# Now, let's set some variables to the filenames where our images are stored.
# We'll be using these a LOT today, so defining them this way does two things. First, it
# saves us a lot of typing. Secondly, if we move our files or want to change which ones we're
# working with, we have to make only one change here and everything below here
# will still work as normal.

#Alex:
#SINGLE_TIFF_PATH = "/Users/alexh/Desktop/20251024/JEM-ARM200FImage_20251024_1518_44_ADF1_ImagePanel1.tif"          # e.g., "data/single/micrograph_01.tif"
#STACK_FOLDER_PATH = "/Users/alexh/Desktop/20251024/JEM-ARM200FImage_20251024_1518_51_ADF1_ImagePanel1"         # e.g., "data/stack_01_frames/"
#OUTPUT_DIR = "outputs_0_2"
#Erik
#SINGLE_TIFF_PATH = "/Users/escott/projects/working-CITEAM/SiGe/JEM-ARM200FImage_20251210_1036_43_ADF1_ImagePanel1/JEM-ARM200FImage_20251210_1036_46_ADF1_1_ImagePanel1.tif"          # e.g., "data/single/micrograph_01.tif"
#STACK_FOLDER_PATH = "/Users/escott/projects/working-CITEAM/SiGe/JEM-ARM200FImage_20251210_1036_43_ADF1_ImagePanel1"         # e.g., "data/stack_01_frames/"
OUTPUT_DIR = "outputs_0_1"
SINGLE_TIFF_PATH = "SiGe/JEM-ARM200FImage_20251210_1036_43_ADF1_ImagePanel1/JEM-ARM200FImage_20251210_1036_46_ADF1_1_ImagePanel1.tif"          # e.g., "data/single/micrograph_01.tif"
STACK_FOLDER_PATH = "SiGe/JEM-ARM200FImage_20251210_1036_43_ADF1_ImagePanel1"         # e.g., "data/stack_01_frames/"

# Others can go here. We'll reduce this to one set of values once we know the exact configuration of
# the lab desktops.

# Create the output directory. If it already exists, that's fine, just go on. If need be,
# create any parent directories.
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

print("done.")

### Reading an image the hard way

**First...**

In day-to-day use, especially on the web, we usually see maybe three image formats: "jpeg", "png", and maybe "gif". Professional graphics users, and scientists, tend to use "tiff". While there are more than two dozen ways to represent data in a tiff file, the most common reason to use that format in the first place is so an image can be stored without any compression. When stored this way we say the format is "lossless".

Among the thousands of packages for Python, some of them handle reading and writing images. We'll use the "tifffile" package to read one of our input files. Using this package is many, many times easier than writing the code to open a tiff file and read everything by hand, stepping through the file as we go and assembling all of that into into something that looks like an image. That would be unpleasant to say the very least.

In this next code cell, we're going to define a function to read some of the parameeters of a TIFF image and return them as a data structure called a "dictionary". 


In [ ]:
# Let's load an image WITHOUT using Hyperspy to get an appreciation for just how much work Hyperspy does for us.
# We'll define a function called "image_stats()" that will collect some basic information about the image and
# return it in JSON format. Then we'll actually load the image, call that function, and print out the results.

def image_stats(img2d):
    img = img2d.astype(float)
    return {
        "shape": list(img.shape),
        "dtype": str(img2d.dtype),
        "min": float(np.min(img)),
        "max": float(np.max(img)),
        "mean": float(np.mean(img)),
        "std": float(np.std(img)),
        "p1": float(np.percentile(img, 1)),
        "p50": float(np.percentile(img, 50)),
        "p99": float(np.percentile(img, 99))
    }

print("done.")

**Then...**

We're ready to actually open the tiff file and print out some information - things like "what are the 1st, 50th, and 99th percentile pixel values".

***More named parameters. They're going to be all over Hyperspy, later.***

In [ ]:
img = tiff.imread(SINGLE_TIFF_PATH)
img = img.astype(float)
print("Single image stats:", json.dumps(image_stats(img), indent=2))

### Displaying our image

It's almost always a good idea to display an image at various points in our workflow if for no other reason than to make sure our code is doing something that plausibly *could* be what we want it to. The following will do nicely.

***cmap='gray' is worth noting***

In [ ]:
# To see the results of our handiwork so far, let's plot that image into our notebook here.
plt.figure(figsize=(5,5))
plt.imshow(img, cmap='gray')
plt.title("0.1.1 Single TIFF")
plt.axis("off")
plt.show()

### And finally, making and saving a thumbnail

Just for fun (and because it's useful) let's make a "thumbnail" image and save it. In fact, let's save it as a "jpeg" image. Jpegs are compressed, sometimes **highly** compressed, and basically unsuitable for doing any science with. On the other hand, they're one of the standard formats for the web. Doing this sort of thing to prepare images for publication isn't unusual at all.

We'll define a function called ```save_jpeg_thmbnail()``` and then we'll use it to produce and save the smaller image.

***Why programmers write function names with () at the end.***

***step through this one with the debugger! This will probably be time consuming - the Python debugger in Jupyter Lab is... challenging.***

***note where the print() output shows up.***

In [ ]:
# And finally, for this section, create a "thumbnail" image and save it as a jpeg (instead of a TIFF file).
# This will create a function called "save_jpeg_thumbnail()" to do the actual work.

def save_jpeg_thumbnail(img2d, out_path, p_low=1, p_high=99, max_size=512):
    img = img2d.astype(float)
    lo, hi = np.percentile(img, [p_low, p_high])
    img = np.clip((img - lo) / (hi - lo + 1e-12), 0, 1)
    H, W = img.shape
    scale = max(H, W) / max_size
    if scale > 1:
        step = int(np.ceil(scale))
        img = img[::step, ::step]
    plt.imsave(out_path, img, cmap='gray')
    
thumb_path = os.path.join(OUTPUT_DIR, Path(SINGLE_TIFF_PATH).stem + "_thumb.jpg")
save_jpeg_thumbnail(img, thumb_path)
print("Saved thumbnail:", thumb_path)

### Doing it the easy way: HyperSpy!

That was a decent amount of work we did to open a file, plot it, and save a thumbnail. Fortunately, there is a package available called "HyperSpy" and we will be using a LOT today. It is made for electron microscopy - originally EELS, but it has been expanded into a more general-purpose package.

Let's do all the work we have to do to read in an image file - in one line of code!

In [ ]:
# Now, let's do this again using Hyperspy. Hyperspy knows how to deal with multiple input file types,
# including HDF5 and the Digital Micrograph variant, and can read literally dozens of image file formats.

print("=========")
print("Loading an image using Hyperspy")
s = hs.load(SINGLE_TIFF_PATH)
print("loaded successfully!")

### Plotting with HyperSpy

HyperSpy provides its own function ```plot()``` for... you guessed it... plotting images. It has quite a few options. A couple of the really useful ones are "cmap" (for color map) and "colorbar" (for whether or not to print the color scale with values).

***Take a look at "s" in the debugger and see what's going on (spoiler alert: it's a list because Digital Micrograph(?) added a thumbnail).***

***Walk through use of the controls on the top left part of the image - zoom, zoom history +/-, home resets the view.***

In [ ]:
#We can play with cmap settings available in hyperspy, one particularly useful one is 'plasma' 

s[0].plot(cmap='plasma')

In [ ]:
# We also do not typically label our axis when looking at realspace images, we can turn that off with the axes_ticks boolean variable (T/F). 
# While we are at it, we can remove the intensity scale using the colorbar boolean

s[0].plot(cmap='plasma', colorbar = False)

### TIFF files, TEM Center, and a detailed look at our files

We're not going to take a deep dive into the TIFF file format and all the ways it has been used (and abused!) over the years, but we are going to take a quick look inside one of our TIFF files and see some of the information stored in there. TIFF allows the storage of a rich collection of metadata. Some of it is universally present and relates the the mechanics of actually storing an image in a computer. Some of it is domain specific. Microscopy is concerned with matters of how the microscope has been set up. Geography, on the other hand, is concerned with longitudes and latitudes. You have to know a little bit about where your image is coming from and how it was produced.

***This is the part where Erik hauls out tiffinfo, shows what's actually in the files, and observes that a file with some metadata in it exists.***

***As an additional bonus, look at the contents of the associated .txt file and especially SM_MICRON_BAR and SM_MICRON_MARKER.***

In [ ]:
# When saving in TEM Center, a thumbnail image is also saved
s[1].plot(colorbar = False)

Take a look at the metadata that HyperSpy is manipulating for us.

In [ ]:
print(s[0].metadata)
print("===============")
print(s[0].original_metadata)

-------------------

## Section 0.1.2 - Histogram and Contrast

In this section, we will plot histograms, inspect intensity distributions, and apply contrast adjustment and optional gamma correction. Here we are looking for areas of interest and/or concern in the collected images - both correcting for non-linear response in both the instruments and in human vision and also detecting and adjusting pixels that are "down in the noise" or "full-scale high".

Let's use the hyperspy function "plot_histograms()" to produce and draw them. We'll experiment with a few different parameters.



In [ ]:
# First, we'll just take the defaults
hs.plot.plot_histograms(s[0])

In [ ]:
# Let's try some different options for binning.

# We could try the square root method (the one Excel uses, BTW)
hs.plot.plot_histograms(s[0], legend=["sqr root binning"], bins="sqrt", max_num_bins=2048)

# Or Knuth's algorithm...
hs.plot.plot_histograms(s[0], legend=["Knuth's binning"], bins="knuth", max_num_bins=2048)

### HyperSpy's plots

HyperSpy naturally has its own built-in functions for plotting. Let's plot our sample image we've been working with and take all of the default settings

***These steps are fairly trivial and we have time. We've treated numpy in a pretty casual way. Time to rectify that.***

In [ ]:
# There are many more binning algorithms available - see https://hyperspy.org/hyperspy-doc/current/reference/api.plot/index.html
# Pop Quiz: do you remember how to set the colormap?

s[0].plot()


In [ ]:
p_low, p_high = 1, 99
lo, hi = np.percentile(s[0].data, [p_low, p_high])
print("lo =", lo, " hi =", hi)
img_cs = np.clip((s[0].data - lo) / (hi - lo + 1e-12), 0, 1)
s[0].data = img_cs

hs.plot.plot_histograms(s[0], legend=["auto binning, clipped"], bins=16)

# Finally, let's adjust the contrast of the image by changing the image's "gamma". This is a mapping from the
# linearity of the detector to the non-linearities of the screen we are looking at combined with the
# non-linearities of human vision. 

gamma = 0.7  # try 0.25, 0.5, 1.0, 1.5, and 2
img_out = np.clip(img_cs ** gamma, 0, 1)
s[0].data = img_out


hs.plot.plot_histograms(s[0], legend=["auto, clipped, gamma"], bins=16)
s[0].plot()    # why might it be a good idea to label all of your plots?

-------------------------

## Section 0.1.3 - Exercise 0.1.3: Noise correction (Gaussian blur)

In this section, we will use some algorithms to remove noise from images (including Gaussian denoising), compare before/after images, and discuss the tradeoff between noise reduction and feature preservation.

**UMD folk: talk about what even IS a Gaussian Filter. I could do it if I had to.**


***1. Deepcopy vs. copy vs. =***

***2. Callback to numpy discussion, above.***

***3. Semicolon syntax***

***4. for loops over lists and sets - "pythonic" is (mostly) "functional"***

***5. scipy is mostly a layer over numpy.***

***6. abstraction is our main tool in computing.***

In [ ]:
# We've messed with the data in "s", specifically s[0], quite a lot. Depending on what we
# might have done during the break, there's a good chance it's a mess. So let's re-load it!

s = hs.load(SINGLE_TIFF_PATH)

import copy
img = copy.deepcopy(s[0].data)

# We'll clip as before to handle the extreme ends of the range
lo, hi = np.percentile(img, [1, 99])
vis = np.clip((img - lo) / (hi - lo + 1e-12), 0, 1)

# Now we'll use SciPy's "ndimage" package to apply a Gaussian filter to the underlying data

sigma = 0.5  # try 0.5, 1.0, 2.0
dn = ndimage.gaussian_filter(vis, sigma=sigma)

plt.figure(figsize=(5,5)); plt.imshow(vis); plt.title("0.1.3 Before"); plt.axis("off"); plt.show()
plt.figure(figsize=(5,5)); plt.imshow(dn); plt.title(f"0.1.3 After Gaussian (sigma={sigma})"); plt.axis("off"); plt.show()

# Let's try successively larger values of sigma - notice the list syntax for for loops
for sigma in [0.25, 0.5, 1.0, 2.0]:
    dn = ndimage.gaussian_filter(vis, sigma=sigma)
    plt.figure(figsize=(5,5)); plt.imshow(dn); plt.title(f"0.1.3 After Gaussian (sigma={sigma})"); plt.axis("off"); plt.show()

### Now, the easier way, with HyperSpy

In [ ]:
# We can implement the noise filtering algorithms of our choice, or we can
# let Hyperspy do the hard work for us...

# start with a clean copy again:

#s = hs.load(SINGLE_TIFF_PATH)

#s[0].change_dtype('float')

#s[0].decomposition()    # taking ALL the defaults - a lot to understand here.

#s[0].plot()


imgStack[0].change_dtype('float64')

imgStack[0].decomposition()    # taking ALL the defaults - a lot to understand here.

imgStack[0].plot()


----------------------

## Section 0.1.4 - Exercise 0.1.4: Drift correction on a stack (cross-correlation registration)

In this section we will identify drift in sequential TEM frames, register a stack, and compare corrected and uncorrected results. This is necessary because as the microscope takes successive frames, there is no chance in the world the images will be perfectly aligned with each other.

"Any sufficiently precise instrument **IS** a thermometer" - some scientist or engineer, once upon a time, somewhere.

**--> Microscopy: physics and math of taking many images and combining them to make one better one. Drift vs. Jitter? Extract structure from motion - not all features move at the same speed.**

**Amateur astronomy, photographing the ISS using an image stack: https://www.skyatnightmagazine.com/advice/how-to-take-a-photo-of-the-iss**

### This entire section needs to be lead by someone who knows image processing for microscopy
### And can speak cogently to working in fourier/frequency space. That isn't Erik. Just saying.

***Take a look at imgStack in the debugger and what it consists of.***

In [ ]:
# The first requirement of drift correction is to have more than one image.
# So far we've just been working with the one, but we have a directory with ten of them.
# Let's use them. It's mildly tedious to do this by hand, but again Hyperspy makes life easy.

SINGLE_TIFF_PATH = "TbFeO3/JEM-ARM200FImage_20251024_1518_51_ADF1_ImagePanel1/JEM-ARM200FImage_20251024_1518_54_ADF1_1_ImagePanel1.tif"          # e.g., "data/single/micrograph_01.tif"
STACK_FOLDER_PATH = "TbFeO3/JEM-ARM200FImage_20251024_1518_51_ADF1_ImagePanel1"         # e.g., "data/stack_01_frames/"
SINGLE_TIFF_METADATA = "TbFeO3/JEM-ARM200FImage_20251024_1518_51_ADF1_ImagePanel1/JEM-ARM200FImage_20251024_1518_54_ADF1_1_ImagePanel1.txt"          # e.g., "data/single/micrograph_01.tif"


tiffFilesWildcard = STACK_FOLDER_PATH + "/*.tif"
print("loading all the files matching", tiffFilesWildcard)
imgStack = hs.load(tiffFilesWildcard, stack=True)

#print(imgStack[0])
#imgStack[0]

print('The type of "imgStack" is', type(imgStack))
print('The length of that list (aka "array") is', len(imgStack))
print('A compact representation of "imgStack" is', imgStack[0])
print("=================")
print("The basic metadata for the first image is:")
print("=================")
print(imgStack[0].metadata)

In [ ]:
# Here you can look at any image in the stack using the slider

imgStack[0].plot(navigator='slider')

In [ ]:
# What's the purpose of the image stack? We can acquire multiple fast exposures and sum them to improve signal-to-noise.
# plot() will normalize, so we get an average really.

summed_image = imgStack[0].sum()
summed_image.plot()
hs.plot.plot_histograms(summed_image,bins='scott')

In [ ]:
# Look! We improved the signal!
# But should we assume each image in the stack is perfectly aligned with the next? (No drift?)

imgStack[0].estimate_shift2D()

In [ ]:
# Based on the above output, it doesn't seem like a safe assumption...
# Let's try to align them using a phase correlation method 
# Link: https://en.wikipedia.org/wiki/Phase_correlation

imgStack[0].align2D()

In [ ]:
#Now see how much estimated shift remains:

imgStack[0].estimate_shift2D()

In [ ]:
# We have a stack of independent images, aligned to each other, but they're still independent images
# until we combine them.

summed_image_aligned = imgStack[0].sum() 
summed_image_aligned.plot()
hs.plot.plot_histograms(summed_image_aligned,bins='scott')

In [ ]:
# By plotting both together, it should be easy to determine which is sharper
hs.plot.plot_images([summed_image,summed_image_aligned])

In [ ]:
# set units and scale for the axes

summed_image_aligned.axes_manager.gui()



In [ ]:
summed_image_aligned.plot()

In [ ]:
# For presentation we sometimes want to "smooth" an image, such as with a gaussian filter
# Note that this can be helpful for presentations, it should not be used for analysis!

from scipy.ndimage import gaussian_filter
s_out = summed_image_aligned.map(gaussian_filter, sigma=1.1, inplace=False)
s_out.plot()
hs.plot.plot_histograms(s_out, bins = 'scott')

In [ ]:
# Noise reduction via HyperSpy's various algorithms, not just a gaussian filter

# Get a fresh copy of the original stack
freshStack = hs.load(tiffFilesWildcard, stack=True)




# it's currently full of 16 bit integers (0 to 65535). We need floating-point values for doing actual math. So we convert:
freshStack[0].change_dtype('float64')

freshStack[0].decomposition()    # taking ALL the defaults - a lot to understand here.

freshStack[0].plot()



### Scale: read the metadata from the text file

In [ ]:
def string_to_float_iterative(s):
    numeric_string = ''.join([c for c in s if c.isdigit() or c == '.'])
    return float(numeric_string)


print(SINGLE_TIFF_METADATA)
with open(SINGLE_TIFF_METADATA) as f:
#with open("/tmp/hi") as f:
    for line in f:
        lineList = line.split()
        if len(lineList)==0:
            break
        if lineList[0] == "$$SM_MICRON_BAR":
            smMicronBar = float(lineList[1])
        if lineList[0] == "$$SM_MICRON_MARKER":
            smMicronMarker = string_to_float_iterative(lineList[1])

try:
    scale=smMicronMarker/smMicronBar
except:
    print("A METADATA PROBLEM HAPPENED!")
    scale=-1000000.0

print ("Metadata text file suggests scale =", scale)

imgStack[0].axes_manager[1].name = "Distance"
imgStack[0].axes_manager[1].units = "nm"
imgStack[0].axes_manager[1].scale = scale
imgStack[0].axes_manager[1].offset = 0

imgStack[0].axes_manager[2].name = "Distance"
imgStack[0].axes_manager[2].units = "nm"
imgStack[0].axes_manager[2].scale = scale
imgStack[0].axes_manager[2].offset = 0

imgStack[0].plot()
print(imgStack[0].calibrate(interactive=True))

### Alternatively, we can set the scale through the interactive controls, use the calibrate() function on a structure of known size, and iterate as needed.

The scale reported in the metadata text file can be off by two percent, so this can be useful.

In [ ]:
imgStack[0].axes_manager.gui()

In [ ]:
imgStack[0].plot()
imgStack[0].calibrate(interactive=True)

## Section 0.1.5 - Measure distances (pixels → nm) interactively

In this section we will measure a feature in pixels, calibrate using the image scale bar, and convert the result into nanometers. There are, as always, several ways to do this. We could measure features in terms of pixels and then convert them to nanometers, or we could tell Hyperspy what the image scale is and have it give us a measurement tool that reads directly in our chosen units. We'll do the latter here.


In [ ]:
# First, we need to open the Hyperspy "axes manager" and enter the values for the scale.
# Run this code cell, and three clickable buttons will appear below: "Axis 1", "Axis 2", and "Axis 3".
# Axis 1 is just the number of images in the stack - 10. Axis 2 is the width axis and axis 3 is the height axis.
# Click on each of those two, enter "nm" for your units, and .0390625000 for the scale. In other words,
# each pixel represents .039-ish nanometers, or 25.6 pixels per nanometer

imgStack[0].axes_manager.gui()

While the GUI has its uses, it's actually going to be easier to read the scale information from the metadata in the .txt files provided by the 'scope. This can then be used to set the scale values of the HyperSpy objects

In [ ]:
# Now we can plot the data. Take a look at the scale mark in the bottom left corner of the image.

imgStack[0].plot()

Once the axes have their scale assigned, we can use the "calibrate" tool to measure distances between features on the images.

In [ ]:
# We don't have to hold a ruler up to the screen. Hyperspy provides a handy "calibrate()" function that lets us
# draw a line and it will report the length in our chose scale units. (If we hadn't set the axes scales above then this
# would just return the number of pixels and we would have to do something sensible with it.
imgStack[0].plot()
imgStack[0].calibrate(interactive=True)

## Section 0.1.6 - FFT Analysis

***I don't know enough to write anything meaninful here.***


In [ ]:
s[0].plot()
line_profile = hs.roi.Line2DROI(400, 250, 220, 600, 100)
line0 = line_profile.interactive(s[0])

In [ ]:
hs.plot.plot_spectra(line0)

In [ ]:
s[0].plot()
roi = hs.roi.RectangularROI()
sliced_signal = roi.interactive(s[0])

In [ ]:
s_fft = hs.interactive(sliced_signal.fft, apodization=True, shift=True, recompute_out_event=None)
s_fft.plot(power_spectrum=True)

In [ ]:
p_roi = hs.roi.PolygonROI([(0.2, 0.4), (0.45, 0.45), (0.45, 0.2), (0.35, 0.35)])
p_roi2 = hs.roi.PolygonROI([(0.1, 0.2), (0.12, 0.2), (0.15, 0.1), (0.2, 0.14)])

In [ ]:
s_fft.plot(power_spectrum=True)
p_roi.add_widget(s_fft, axes=s_fft.axes_manager.signal_axes)
p_roi2.add_widget(s_fft, axes=s_fft.axes_manager.signal_axes)

In [ ]:
s_roi_combined = hs.roi.combine_rois(s_fft, [p_roi, p_roi2])
s_roi_combined.plot()

In [ ]:
boolean_mask = hs.roi.mask_from_rois([p_roi, p_roi2], s_fft.axes_manager)
boolean_mask = hs.signals.Signal2D(boolean_mask)
boolean_mask.plot()

In [ ]:
np.shape(s_fft) == np.shape(boolean_mask)

In [ ]:
# This will error out, perhaps because the smaller N-gon is folded?

masked_signal = s_fft*boolean_mask

In [ ]:
masked_signal.plot(power_spectrum=True)

In [ ]:
im_ifft = masked_signal.ifft()
im_ifft.plot()